In [28]:
df = df[df["tourney_id"].ne("tourney_id")].copy()

# Strip whitespace in all object columns
obj_cols = df.select_dtypes(include="object").columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())

df_qual = df_qual[df_qual["tourney_id"].ne("tourney_id")].copy()

# Strip whitespace in all object columns
obj_cols = df_qual.select_dtypes(include="object").columns
df_qual[obj_cols] = df_qual[obj_cols].apply(lambda s: s.str.strip())

In [29]:
# Parse tournament date (YYYYMMDD)

# Columns that should be numeric (this dataset has many)
numeric_cols = [
    "draw_size", "match_num", "best_of", "minutes",
    "winner_id", "winner_seed", "winner_ht", "winner_age",
    "loser_id", "loser_seed", "loser_ht", "loser_age",
    "winner_rank", "winner_rank_points", "loser_rank", "loser_rank_points",
    # post-match stats (keep numeric even if you later drop for leakage)
    "w_ace","w_df","w_svpt","w_1stIn","w_1stWon","w_2ndWon","w_SvGms","w_bpSaved","w_bpFaced",
    "l_ace","l_df","l_svpt","l_1stIn","l_1stWon","l_2ndWon","l_SvGms","l_bpSaved","l_bpFaced", "match_date",
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        
        
df["surface"] = df["surface"].replace({"Carpet": "Hard"})


In [30]:
df_qual = df_qual.drop('match_date', axis=1)
df_qual = df_qual.rename(columns={'match_date_yyyymmdd': 'match_date'})
for c in numeric_cols:
    if c in df_qual.columns:
        df_qual[c] = pd.to_numeric(df_qual[c], errors="coerce")
        
        
df_qual["surface"] = df_qual["surface"].replace({"Carpet": "Hard"})
df_qual["score"] = df_qual["score"].fillna("ddd")
df_qual["score"].isna().sum()

np.int64(0)

In [31]:
df = pd.concat([df, df_qual], ignore_index=True)
df = df.sort_values("tourney_date", kind="mergesort").reset_index(drop=True)
df["score"].isna().sum()
df["score"].isna().sum()
print(df["score"].isna().sum())
print(df["tourney_date"])

0
0        20170102
1        20170102
2        20170102
3        20170102
4        20170102
           ...   
53362    20260601
53363    20260601
53364    20260601
53365    20260601
53366    20260601
Name: tourney_date, Length: 53367, dtype: int64


In [32]:
# -------- Player bio cleanup --------
# Treat impossible heights as missing before imputation.
HEIGHT_COLS = ["winner_ht", "loser_ht"]
HEIGHT_MIN_CM, HEIGHT_MAX_CM = 140, 220
for col in HEIGHT_COLS:
    df.loc[~df[col].between(HEIGHT_MIN_CM, HEIGHT_MAX_CM), col] = np.nan

# Keep ffill + bfill inside each player group using transform(...).
def fill_player_numeric(frame, player_col, value_col):
    frame[value_col] = frame.groupby(player_col)[value_col].transform(lambda s: s.ffill().bfill())
    frame[value_col] = frame[value_col].fillna(frame[value_col].median())

for value_col in ["winner_age", "winner_ht"]:
    fill_player_numeric(df, "winner_name", value_col)

for value_col in ["loser_age", "loser_ht"]:
    fill_player_numeric(df, "loser_name", value_col)

# Fill missing handedness with mode; fallback to right-handed if empty.
for col in ["winner_hand", "loser_hand"]:
    mode = df[col].mode(dropna=True)
    fill_val = mode.iloc[0] if not mode.empty else "R"
    df[col] = df[col].fillna(fill_val)

# -------- Match stat cleanup --------
# Fill post-match stats within best_of groups, then fallback to global median.
POST_MATCH_STATS = [
    "ace", "df", "svpt", "1stIn", "1stWon",
    "2ndWon", "bpSaved", "bpFaced", "SvGms"
]
for stat in POST_MATCH_STATS:
    for prefix in ["w_", "l_"]:
        col = f"{prefix}{stat}"
        if col in df.columns:
            df[col] = df.groupby("best_of")[col].transform(lambda s: s.fillna(s.median()))
            df[col] = df[col].fillna(df[col].median())

# Match duration and categorical fallback.
df["minutes"] = df.groupby("best_of")["minutes"].transform(lambda s: s.fillna(s.median()))
df["surface"] = df["surface"].fillna("Hard")

# Rank points: use nearest known value within each player history.
for player_col, rank_col in [
    ("winner_name", "winner_rank_points"),
    ("loser_name", "loser_rank_points"),
]:
    df[rank_col] = df.groupby(player_col)[rank_col].transform(lambda s: s.ffill().bfill())


In [33]:
# No missing values left
missing_summary = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .to_frame("missing_frac")
)

missing_summary

,missing_frac
winner_entry,0.841344
loser_entry,0.739521
loser_seed,0.715723
winner_seed,0.529784
winner_id,0.362171
loser_id,0.362115
match_day_offset,0.352709
match_date,0.352709
loser_rank,0.019282
loser_rank_points,0.011093


In [34]:
def test_cleaned_df(df):
    """Sanity-check the cleaned dataframe before export."""
    required_cols = [
        "tourney_id", "tourney_date", "surface", "winner_name", "loser_name",
        "best_of", "score", "winner_age", "loser_age", "winner_ht", "loser_ht",
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    assert not missing_cols, f"Missing expected columns: {missing_cols}"

    # Key columns should have no nulls after cleaning
    for col in required_cols:
        n_missing = df[col].isna().sum()
        assert n_missing == 0, f"{col} has {n_missing} missing values"

    assert set(df["surface"].unique()) <= {"Hard", "Clay", "Grass"}, "Unexpected surface value"
    assert (df["best_of"].isin([3, 5])).all(), "best_of contains values other than 3 or 5"
    assert len(df) > 0, "Cleaned dataframe is empty"

    print(f"All checks passed on {len(df)} rows.")


test_cleaned_df(df)


All checks passed on 53367 rows.


In [35]:
df.to_csv('atp_matches_data_cleaned.csv', index=False, header=True)